# Training Helpers Validation Tutorial

This notebook checks the optional supervised training helpers:

$$
\text{model},\ \text{loader},\ \text{loss},\ \text{optimizer}
\quad\longrightarrow\quad
\text{history},\ \text{metric},\ \text{checkpoint}.
$$

The helpers do not define a SILVA architecture by themselves. They provide the
repeatable training loop around any PyTorch model, including models built from
`silva_networks`.

<!-- silva-numbered-citations:start -->
**Numbered literature:** [[1]](https://jseluis.github.io/silva-networks/paper/references/#ref-1), [[4]](https://jseluis.github.io/silva-networks/paper/references/#ref-4), [[39]](https://jseluis.github.io/silva-networks/paper/references/#ref-39). Each number opens the complete citation and its primary external source.
<!-- silva-numbered-citations:end -->


In [1]:
from pathlib import Path
import importlib.util
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/jseluis/silva-networks.git"

def find_local_silva_root():
    candidates = [
        Path.cwd(),
        Path("/content/silva-networks"),
        Path("/content/drive/MyDrive/silva-networks"),
    ]
    root = Path.cwd()
    while root != root.parent:
        candidates.append(root)
        root = root.parent
    for candidate in candidates:
        if (candidate / "src" / "silva_networks").exists():
            return candidate
    return None

root = find_local_silva_root()
if root is not None:
    sys.path.insert(0, str(root / "src"))
elif IN_COLAB and importlib.util.find_spec("silva_networks") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", f"git+{REPO_URL}"])
    root = Path.cwd()
else:
    root = Path.cwd()

In [2]:
from pathlib import Path
import tempfile

import torch
from torch.utils.data import DataLoader, TensorDataset

from silva_networks import (
    TrainConfig,
    evaluate,
    fit_supervised,
    resolve_device,
    seed_everything,
)

device = resolve_device("cuda" if torch.cuda.is_available() else "cpu")
seed_everything(10)
device

device(type='cpu')

## Synthetic Classification Data

Use a tiny linearly separable dataset. The batch contract here is the ordinary
PyTorch `(x, y)` tuple:

$$
x\in\mathbb R^{N\times d},
\qquad
y\in\{0,1\}^N.
$$

In [3]:
x_pos = torch.randn(24, 3) + torch.tensor([1.5, 0.0, 0.0])
x_neg = torch.randn(24, 3) + torch.tensor([-1.5, 0.0, 0.0])
x = torch.cat([x_pos, x_neg], dim=0)
y = torch.cat([torch.ones(24, dtype=torch.long), torch.zeros(24, dtype=torch.long)])

loader = DataLoader(TensorDataset(x, y), batch_size=12, shuffle=True)
val_loader = DataLoader(TensorDataset(x, y), batch_size=16)
model = torch.nn.Sequential(
    torch.nn.Linear(3, 12),
    torch.nn.Tanh(),
    torch.nn.Linear(12, 2),
)

## Fit and Evaluate

For classification, `loss="auto"` becomes cross entropy and `metric="auto"`
becomes accuracy. The training helper moves the model and batches to the
requested device.

In [4]:
with tempfile.TemporaryDirectory() as tmp:
    checkpoint = Path(tmp) / "training.pt"
    result = fit_supervised(
        model,
        loader,
        val_loader,
        config=TrainConfig(
            task="classification",
            epochs=3,
            lr=0.05,
            optimizer="adam",
            gradient_clipping=1.0,
            device=device,
            seed=10,
            checkpoint_path=checkpoint,
        ),
    )
    evaluation = evaluate(model, val_loader, device=device)
    print("epochs:", len(result.history))
    print("best epoch:", result.best_epoch)
    print("metric:", evaluation.metric_name, round(evaluation.metric, 4))
    print("checkpoint exists:", checkpoint.exists())

epochs: 3
best epoch: 2
metric: accuracy 0.9792
checkpoint exists: True


## Resume

When `resume=True`, the helper reloads the model, optimizer, scheduler if
present, and history from the checkpoint path.

In [5]:
with tempfile.TemporaryDirectory() as tmp:
    checkpoint = Path(tmp) / "resume.pt"
    _ = fit_supervised(
        model,
        loader,
        config=TrainConfig(epochs=1, lr=0.01, device=device, checkpoint_path=checkpoint),
    )
    resumed = fit_supervised(
        model,
        loader,
        config=TrainConfig(epochs=2, lr=0.01, device=device, checkpoint_path=checkpoint, resume=True),
    )
    print("history after resume:", len(resumed.history))

history after resume: 2


## Citation

If this notebook or package is used, cite:

```text
Dr. Jose Luis Silva. SILVA Networks. Version 1.1.0. MIT License.
https://github.com/jseluis/silva-networks
https://doi.org/10.5281/zenodo.21770098
```

When training SILVA models in connection with the SILVA Networks paper, cite
the paper as well.

## From 10 Training Helpers Smoke to a Custom SILVA Family

The construction in this notebook can be separated into the universal
conditioned-equilibrium contract

$$
z_0=I_\eta(x),\qquad
z^\star=T_\theta(z^\star,x),\qquad
\widehat y=Q_\psi(z^\star).
$$

For this topic:

| Part | Concrete interpretation |
| --- | --- |
| Equilibrium state | the tensor solved to equilibrium |
| Condition | the observed input or source tensor |
| Repeated computation | the state-preserving transition evaluated by the root solver |
| Required invariants | shape, device, dtype, finiteness, and differentiability |
| Replaceable components | initializer, source encoder, transition, readout, and solver |

The initializer and source path are evaluated outside or alongside the root
solve. Only the state-preserving transition is repeated. Replacing an internal
architecture does not change this equation, provided the transition still maps
the same state space into itself.


In [6]:
import torch as silva_extension_torch
from torch import nn as silva_extension_nn

from silva_networks import (
    SILVAConditionedEquilibrium,
    SILVAZeroInitializer,
    SolverConfig,
    validate_silva_transition,
)


class NotebookExtensionTransition(silva_extension_nn.Module):
    def __init__(self, condition_dim=2, state_dim=3):
        super().__init__()
        self.source = silva_extension_nn.Linear(condition_dim, state_dim)
        self.state_field = silva_extension_nn.Sequential(
            silva_extension_nn.Linear(state_dim, 2 * state_dim),
            silva_extension_nn.Tanh(),
            silva_extension_nn.Linear(2 * state_dim, state_dim),
        )

    def forward(self, state, condition):
        return silva_extension_torch.tanh(
            self.source(condition) + 0.15 * self.state_field(state)
        )


silva_extension_torch.manual_seed(610)
notebook_condition = silva_extension_torch.linspace(-1.0, 1.0, 8).reshape(4, 2)
notebook_state0 = silva_extension_torch.zeros(4, 3)
notebook_transition = NotebookExtensionTransition()

notebook_report = validate_silva_transition(
    notebook_transition,
    notebook_state0,
    notebook_condition,
)
assert notebook_report.valid

with silva_extension_torch.no_grad():
    notebook_reference_step = silva_extension_torch.tanh(
        notebook_transition.source(notebook_condition)
        + 0.15 * notebook_transition.state_field(notebook_state0)
    )
silva_extension_torch.testing.assert_close(
    notebook_transition(notebook_state0, notebook_condition),
    notebook_reference_step,
)

notebook_custom_model = SILVAConditionedEquilibrium(
    notebook_transition,
    SILVAZeroInitializer(3),
    readout=silva_extension_nn.Linear(3, 1),
    config=SolverConfig(
        solver="picard",
        max_iter=40,
        tol=1e-7,
        backward_mode="implicit",
        backward_solver="gmres",
        anderson_batch_dims=1,
    ),
)
notebook_custom_result = notebook_custom_model(
    notebook_condition,
    return_result=True,
)
assert notebook_custom_result.output.shape == (4, 1)
assert notebook_custom_result.solver_result.residual < 1e-5

notebook_custom_result.output.square().mean().backward()
assert all(
    parameter.grad is not None and silva_extension_torch.isfinite(parameter.grad).all()
    for parameter in notebook_custom_model.parameters()
)
print("custom transition:", notebook_report)
print("equilibrium residual:", notebook_custom_result.solver_result.residual)


custom transition: SILVATransitionReport(state_shape=(4, 3), output_shape=(4, 3), preserves_shape=True, preserves_device=True, preserves_dtype=True, finite=True, differentiable=True, parameter_count=54)
equilibrium residual: 5.960464477539063e-08


## Numerical Equivalence, Compact Reproduction, and Scale

Before training, compare one packaged transition with an independently written
update:

$$
e_{\mathrm{step}}
=\frac{\|T_\theta(z,x)-T_{\mathrm{ref}}(z,x)\|_2}
{\|T_{\mathrm{ref}}(z,x)\|_2+\varepsilon}.
$$

After solving, report the fixed-point residual separately:

$$
e_{\mathrm{fp}}
=\frac{\|T_\theta(z^\star,x)-z^\star\|_2}
{\|z^\star\|_2+\varepsilon}.
$$

For this notebook, a compact reproduction must declare and assert
**fixed-point residual and task error against a deterministic target**. A full experiment must additionally record the
source dataset version and split, preprocessing, architecture widths, solver
and optimizer schedules, random seeds, baseline configuration, checkpoints,
and every deviation from the cited protocol.

The principal scaling axes are **state width, batch size, and data volume**. Increase one axis at
a time, retain the compact deterministic case as a regression test, and record
task error, domain-specific residual, forward residual, backward linear
residual, memory use, and runtime independently.

### Extension Exercises

1. Replace one component from this notebook while preserving its state and
   domain invariants.
2. Write the replacement first as an independent reference function, then as
   a module, and assert one-step equivalence.
3. Compare two solver configurations on the identical trained transition.
4. Add a compact baseline and a predeclared metric threshold.
5. Create a full-scale configuration without weakening the compact tests.

The complete authoring protocol is documented in
[Extending SILVA](https://jseluis.github.io/silva-networks/learn/extending-silva/).


In [7]:
notebook_reproduction_record = {
    "notebook": '10_training_helpers_smoke.ipynb',
    "state": 'the tensor solved to equilibrium',
    "condition": 'the observed input or source tensor',
    "transition": 'the state-preserving transition evaluated by the root solver',
    "invariants": 'shape, device, dtype, finiteness, and differentiability',
    "compact_metric": 'fixed-point residual and task error against a deterministic target',
    "scale_axis": 'state width, batch size, and data volume',
}
assert all(notebook_reproduction_record.values())
notebook_reproduction_record


{'notebook': '10_training_helpers_smoke.ipynb',
 'state': 'the tensor solved to equilibrium',
 'condition': 'the observed input or source tensor',
 'transition': 'the state-preserving transition evaluated by the root solver',
 'invariants': 'shape, device, dtype, finiteness, and differentiability',
 'compact_metric': 'fixed-point residual and task error against a deterministic target',
 'scale_axis': 'state width, batch size, and data volume'}

## Where to Go Next

| Question | Page |
| --- | --- |
| Which training objects and result fields are public? | [Training API](https://jseluis.github.io/silva-networks/api/training/) |
| What evidence should a trained experiment report? | [Reconstructing Paper Experiments](https://jseluis.github.io/silva-networks/learn/reconstructing-paper-experiments/) |
| Which measured outputs are currently published? | [Results](https://jseluis.github.io/silva-networks/results/) |
